# Multimodal Sentiment Analysis & Conflict Detection
## Food Sector — Yelp Dataset
**Course:** Machine Learning  
**Group Members:** Ahmet Kaan Kavaz & Kutlay Kavkas

---
### Pipeline Overview
1. Data Collection (Yelp food reviews + photos)
2. Data Cleaning & Missing Value Control
3. Outlier & Noise Analysis
4. Feature Scaling & Normalization
5. Dual-Stream Encoding (RoBERTa + ViT)
6. Feature Fusion Layer
7. Model Training & Hyperparameter Tuning
8. Performance Evaluation

## Install Dependencies

In [ ]:
!pip install transformers torch torchvision pillow scikit-learn matplotlib seaborn tqdm -q

## Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
import zipfile, os
drive.mount('/content/drive')

# PATHS — change these if your Drive folder structure is different
BUSINESS_JSON = '/content/drive/MyDrive/yelp_data/yelp_academic_dataset_business.json'
REVIEW_JSON   = '/content/drive/MyDrive/yelp_data/yelp_academic_dataset_review.json'
PHOTOS_JSON   = '/content/drive/MyDrive/yelp_data/photos.json'
PHOTOS_ZIP    = '/content/drive/MyDrive/yelp_data/photos.zip'
PHOTOS_DIR    = '/content/photos/'

# Unzip photos only if not already extracted
if not os.path.exists(PHOTOS_DIR):
    print('Extracting photos.zip ...')
    with zipfile.ZipFile(PHOTOS_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Extraction complete')
else:
    print('Photos already extracted')

print('Paths configured')

## Imports

In [ ]:
import os, json, re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import (
    RobertaTokenizer, RobertaModel,
    ViTModel
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

---
## STEP 1 — Data Collection
Extract food/restaurant businesses from Yelp, then join reviews and photos.

In [ ]:
FOOD_CATEGORIES = [
    'restaurants', 'food', 'pizza', 'burger', 'sushi',
    'breakfast', 'brunch', 'bakeries', 'desserts', 'coffee'
]

food_business_ids = set()
with open(BUSINESS_JSON, 'r', encoding='utf-8') as f:
    for line in f:
        biz = json.loads(line)
        cats = (biz.get('categories') or '').lower()
        if any(cat in cats for cat in FOOD_CATEGORIES):
            food_business_ids.add(biz['business_id'])

print(f'Food businesses found: {len(food_business_ids):,}')

MAX_REVIEWS = 3_000
reviews = []
with open(REVIEW_JSON, 'r', encoding='utf-8') as f:
    for line in f:
        if len(reviews) >= MAX_REVIEWS:
            break
        r = json.loads(line)
        if r['business_id'] in food_business_ids:
            reviews.append({
                'review_id':   r['review_id'],
                'business_id': r['business_id'],
                'text':        r['text'],
                'stars':       r['stars']
            })

df_reviews = pd.DataFrame(reviews)
print(f'Food reviews loaded: {len(df_reviews):,}')

photos = []
with open(PHOTOS_JSON, 'r', encoding='utf-8') as f:
    for line in f:
        p = json.loads(line)
        if p.get('label') == 'food' and p['business_id'] in food_business_ids:
            photos.append({
                'photo_id':    p['photo_id'],
                'business_id': p['business_id'],
                'label':       p['label']
            })

df_photos = pd.DataFrame(photos)
print(f'Food photos loaded: {len(df_photos):,}')

df_merged = df_reviews.merge(df_photos, on='business_id', how='inner')
df_merged = df_merged.drop_duplicates(subset=['review_id'])
print(f'Paired (text + image) samples: {len(df_merged):,}')
df_merged.head()

---
## STEP 2 — Data Cleaning & Missing Value Control
Filter incomplete reviews, missing images, and corrupted files.

In [ ]:
print('=== Missing Value Report (before cleaning) ===')
print(df_merged.isnull().sum())

df_clean = df_merged.dropna(subset=['text', 'stars', 'photo_id'])
df_clean = df_clean[df_clean['text'].str.strip().str.len() > 0]

def photo_exists(photo_id):
    return os.path.isfile(os.path.join(PHOTOS_DIR, photo_id + '.jpg'))

tqdm.pandas(desc='Checking photo files')
df_clean = df_clean[df_clean['photo_id'].progress_apply(photo_exists)]

def is_valid_image(photo_id):
    try:
        img = Image.open(os.path.join(PHOTOS_DIR, photo_id + '.jpg'))
        img.verify()
        return True
    except Exception:
        return False

df_clean = df_clean[df_clean['photo_id'].progress_apply(is_valid_image)]
df_clean = df_clean.reset_index(drop=True)

print(f'Samples after cleaning: {len(df_clean):,}')
print('\n=== Missing Value Report (after cleaning) ===')
print(df_clean.isnull().sum())

---
## STEP 3 — Outlier & Noise Analysis
Handle extreme text lengths and low-quality photos.

In [ ]:
df_clean['text_length'] = df_clean['text'].apply(len)

print('=== Text Length Statistics ===')
print(df_clean['text_length'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_clean['text_length'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Text Length Distribution (before filtering)')
axes[0].set_xlabel('Character count')

Q1 = df_clean['text_length'].quantile(0.05)
Q3 = df_clean['text_length'].quantile(0.95)
df_clean = df_clean[(df_clean['text_length'] >= Q1) & (df_clean['text_length'] <= Q3)]
print(f'Samples after text-length outlier removal: {len(df_clean):,}')

axes[1].hist(df_clean['text_length'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Text Length Distribution (after filtering)')
axes[1].set_xlabel('Character count')
plt.tight_layout()
plt.show()

def get_image_size(photo_id):
    try:
        img = Image.open(os.path.join(PHOTOS_DIR, photo_id + '.jpg'))
        return img.size
    except:
        return (0, 0)

df_clean[['img_w', 'img_h']] = pd.DataFrame(
    df_clean['photo_id'].apply(get_image_size).tolist(),
    index=df_clean.index
)

MIN_SIZE = 64
before = len(df_clean)
df_clean = df_clean[(df_clean['img_w'] >= MIN_SIZE) & (df_clean['img_h'] >= MIN_SIZE)]
print(f'Removed {before - len(df_clean)} images below {MIN_SIZE}x{MIN_SIZE}px')
print(f'Remaining samples: {len(df_clean):,}')
df_clean = df_clean.reset_index(drop=True)

---
## STEP 4 — Feature Scaling & Normalization
- Label mapping: 1-2 stars = Negative, 3 = Neutral, 4-5 = Positive
- Text tokenization via RoBERTa
- Image normalization & resizing for ViT (224x224)

In [ ]:
def stars_to_label(stars):
    if stars <= 2:   return 0  # Negative
    elif stars == 3: return 1  # Neutral
    else:            return 2  # Positive

df_clean['label'] = df_clean['stars'].apply(stars_to_label)
LABEL_NAMES = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print('=== Class Distribution ===')
print(df_clean['label'].value_counts().rename(LABEL_NAMES))

plt.figure(figsize=(6,4))
sns.countplot(x=df_clean['label'].map(LABEL_NAMES), palette=['#e74c3c','#f39c12','#2ecc71'])
plt.title('Sentiment Class Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

TEXT_MAX_LEN = 128
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
print('RoBERTa tokenizer loaded')

vit_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
print('Image transform configured')

---
## STEP 5 — Dataset Class & Train/Val/Test Split (70/15/15)

In [ ]:
class YelpFoodDataset(Dataset):
    def __init__(self, dataframe, tokenizer, transform, photos_dir, max_len=128):
        self.df         = dataframe.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.transform  = transform
        self.photos_dir = photos_dir
        self.max_len    = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            row['text'],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)

        img_path = os.path.join(self.photos_dir, row['photo_id'] + '.jpg')
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        label = torch.tensor(row['label'], dtype=torch.long)
        return input_ids, attention_mask, image, label


train_df, temp_df = train_test_split(df_clean, test_size=0.30, random_state=SEED, stratify=df_clean['label'])
val_df,   test_df = train_test_split(temp_df,  test_size=0.50, random_state=SEED, stratify=temp_df['label'])

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

BATCH_SIZE = 8
train_dataset = YelpFoodDataset(train_df, roberta_tokenizer, vit_transform, PHOTOS_DIR, TEXT_MAX_LEN)
val_dataset   = YelpFoodDataset(val_df,   roberta_tokenizer, vit_transform, PHOTOS_DIR, TEXT_MAX_LEN)
test_dataset  = YelpFoodDataset(test_df,  roberta_tokenizer, vit_transform, PHOTOS_DIR, TEXT_MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('DataLoaders ready')

---
## STEP 6 — Dual-Stream Encoding + Feature Fusion Layer
**RoBERTa** (NLP stream) + **ViT** (Vision stream) → Concatenation → MLP Classifier

In [ ]:
class MultimodalSentimentModel(nn.Module):
    """
    Dual-Stream Architecture:
      NLP Stream   : RoBERTa-base  -> 768-dim CLS embedding
      Vision Stream: ViT-base/16   -> 768-dim CLS embedding
      Fusion       : Concatenation -> 1536-dim vector
      Classifier   : MLP (1536 -> 512 -> 128 -> num_classes)
    """
    def __init__(self, num_classes=3, dropout=0.3):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained('roberta-base')
        self.vit        = ViTModel.from_pretrained('google/vit-base-patch16-224')
        text_dim  = self.roberta.config.hidden_size   # 768
        image_dim = self.vit.config.hidden_size        # 768
        fused_dim = text_dim + image_dim               # 1536
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        text_output  = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        text_embed   = text_output.last_hidden_state[:, 0, :]
        image_output = self.vit(pixel_values=pixel_values)
        image_embed  = image_output.last_hidden_state[:, 0, :]
        fused  = torch.cat([text_embed, image_embed], dim=1)
        logits = self.classifier(fused)
        return logits


model = MultimodalSentimentModel(num_classes=3).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(model)

---
## STEP 7 — Model Training & Hyperparameter Tuning

In [ ]:
from transformers import get_linear_schedule_with_warmup

LEARNING_RATE = 2e-5
NUM_EPOCHS    = 5
WEIGHT_DECAY  = 0.01

optimizer     = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion     = nn.CrossEntropyLoss()
total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(0.1 * total_steps)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_f1  = 0.0
best_model_path = '/content/best_model.pt'

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for input_ids, attention_mask, images, labels in tqdm(loader, leave=False):
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            images         = images.to(DEVICE)
            labels         = labels.to(DEVICE)
            logits = model(input_ids, attention_mask, images)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return avg_loss, acc, f1

print('Starting training...\n')
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, train_f1 = run_epoch(train_loader, train=True)
    val_loss,   val_acc,   val_f1   = run_epoch(val_loader,   train=False)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    print(f'Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)
        print(f'  -> Best model saved (Val F1: {best_val_f1:.4f})')

print('\nTraining complete')

### Training History Plots

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='tomato')
axes[0].plot(epochs_range, history['val_loss'],   'o-', label='Val Loss',   color='steelblue')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(epochs_range, history['val_acc'], 'o-', color='seagreen')
axes[1].set_title('Validation Accuracy'); axes[1].set_xlabel('Epoch')
axes[2].plot(epochs_range, history['val_f1'], 'o-', color='darkorchid')
axes[2].set_title('Validation F1 Score'); axes[2].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

---
## STEP 8 — Performance Evaluation
Accuracy, F1-Score, Confusion Matrix, and Consistency Metric (conflict detection)

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_text_preds, all_image_preds = [], [], [], []

with torch.no_grad():
    for input_ids, attention_mask, images, labels in tqdm(test_loader, desc='Evaluating'):
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        images         = images.to(DEVICE)
        labels         = labels.to(DEVICE)

        logits = model(input_ids, attention_mask, images)
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        # Text-only stream
        text_out   = model.roberta(input_ids=input_ids, attention_mask=attention_mask)
        text_embed = text_out.last_hidden_state[:, 0, :]
        zero_img   = torch.zeros_like(text_embed)
        text_logits = model.classifier(torch.cat([text_embed, zero_img], dim=1))
        all_text_preds.extend(text_logits.argmax(dim=1).cpu().numpy())

        # Image-only stream
        img_out    = model.vit(pixel_values=images)
        img_embed  = img_out.last_hidden_state[:, 0, :]
        zero_text  = torch.zeros_like(img_embed)
        img_logits = model.classifier(torch.cat([zero_text, img_embed], dim=1))
        all_image_preds.extend(img_logits.argmax(dim=1).cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='weighted')
print(f'\n=== TEST SET RESULTS ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=list(LABEL_NAMES.values())))

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES.values(),
            yticklabels=LABEL_NAMES.values())
plt.title('Confusion Matrix - Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
all_text_preds  = np.array(all_text_preds)
all_image_preds = np.array(all_image_preds)
all_preds_arr   = np.array(all_preds)
all_labels_arr  = np.array(all_labels)

conflicts = (all_text_preds != all_image_preds)
consistency_score = 1 - conflicts.mean()

print(f'=== CONSISTENCY METRIC (Conflict Detection) ===')
print(f'Total test samples : {len(conflicts):,}')
print(f'Conflicts detected : {conflicts.sum():,} ({conflicts.mean()*100:.1f}%)')
print(f'Consistency Score  : {consistency_score:.4f}')

acc_conflict    = accuracy_score(all_labels_arr[conflicts],  all_preds_arr[conflicts])  if conflicts.sum() > 0 else float('nan')
acc_no_conflict = accuracy_score(all_labels_arr[~conflicts], all_preds_arr[~conflicts]) if (~conflicts).sum() > 0 else float('nan')
print(f'\nAccuracy on conflicting samples    : {acc_conflict:.4f}')
print(f'Accuracy on non-conflicting samples: {acc_no_conflict:.4f}')

plt.figure(figsize=(6, 4))
plt.bar(['Non-conflict', 'Conflict'], [(~conflicts).sum(), conflicts.sum()], color=['seagreen', 'tomato'])
plt.title('Modality Conflict Analysis')
plt.ylabel('Number of Samples')
plt.tight_layout()
plt.savefig('conflict_analysis.png', dpi=150)
plt.show()

---
## Final Summary

In [ ]:
print('=' * 50)
print('  MULTIMODAL SENTIMENT ANALYSIS - RESULTS')
print('=' * 50)
print(f'  Dataset         : Yelp Food Reviews')
print(f'  Text Encoder    : RoBERTa-base')
print(f'  Image Encoder   : ViT-base-patch16-224')
print(f'  Fusion Strategy : Late Fusion (Concatenation)')
print(f'  Classes         : Negative / Neutral / Positive')
print('-' * 50)
print(f'  Test Accuracy   : {acc:.4f}')
print(f'  Test F1-Score   : {f1:.4f}')
print(f'  Consistency     : {consistency_score:.4f}')
print(f'  Conflicts       : {conflicts.sum():,} / {len(conflicts):,} ({conflicts.mean()*100:.1f}%)')
print('=' * 50)

---
## DEMO: Canli Tahmin
Metin ve fotograf vererek modeli canli test edin.

In [ ]:
# DEMO: Tek bir review icin canli tahmin
from PIL import Image
import requests
from io import BytesIO

def predict(text, image_source):
    model.eval()

    # Text encoding
    encoding = roberta_tokenizer(
        text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    # Image loading
    if image_source.startswith('http'):
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(image_source, headers=headers, timeout=10)
        image = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        image = Image.open(image_source).convert('RGB')

    image_tensor = vit_transform(image).unsqueeze(0).to(DEVICE)

    # Prediction
    with torch.no_grad():
        logits = model(input_ids, attention_mask, image_tensor)
        probs  = torch.softmax(logits, dim=1)[0]
        pred   = logits.argmax(dim=1).item()

    label_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}

    print('=' * 45)
    print('      SENTIMENT ANALYSIS RESULT')
    print('=' * 45)
    short = text[:80] + '...' if len(text) > 80 else text
    print(f'  Review : {short}')
    print(f'  Result : {label_map[pred]}')
    print('-' * 45)
    print(f'  Negative : {probs[0]*100:.1f}%')
    print(f'  Neutral  : {probs[1]*100:.1f}%')
    print(f'  Positive : {probs[2]*100:.1f}%')
    print('=' * 45)

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(label_map[pred], fontsize=16)
    plt.tight_layout()
    plt.show()


# Ornek 1: Pozitif review
predict(
    text='Absolutely amazing food! The pizza was perfectly crispy and the toppings were fresh. Best meal I have had in years. Will definitely come back!',
    image_source='https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=400'
)

# Ornek 2: Negatif review
predict(
    text='Terrible experience. The food was cold, the service was rude and we waited over an hour. Never coming back to this place.',
    image_source='https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=400'
)
